# 第11周｜亚组与数据质量实验

学习目标：亚组样本量、事件数、分布变化、敏感性。

__数据：默认使用官方 PRIME-CVD 5 万人数据__（`python run.py download` 下载，来源与 MD5 核验记录在 `data/raw/manifest.json`）。把 `MODE` 改成 `"demo"` 可以切换到离线演示数据，但演示数据是另外生成的，**不是 PRIME-CVD**，结果不能当作官方结果。

本周验收：区分受控数据错误实验、亚组探索和外部验证。

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

# 从项目根目录、notebooks目录或其他项目内目录启动均可。
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run.py").exists()), None)
if ROOT is None:
    raise RuntimeError("请在解压后的项目目录中启动 Jupyter。")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from primecvd.common import load_config, provenance_label
from primecvd.pipeline import prepare
from primecvd.splitting import select_split

MODE = "official"  # 官方数据；离线练习或跑测试时可改为 "demo"（演示数据不是 PRIME-CVD）
cfg = load_config(ROOT / "configs" / f"{MODE}.json")
cfg["output_dir"] = f"outputs/notebooks_{MODE}"
print(provenance_label(cfg))
df, splits = prepare(cfg)
train = select_split(df, splits, "train")
valid = select_split(df, splits, "valid")
# 不创建test DataFrame；模型试验只使用train和valid。
print("队列、训练集、验证集：", len(df), len(train), len(valid))

# 命令行绘图模块采用无窗口后端；在Notebook中显式启用行内图像。
get_ipython().run_line_magic("matplotlib", "inline")
from primecvd import pubplot as pp   # 与精讲篇相同的顶刊作图风格（在启用行内图像之后设置）
pp.setup()

OFFICIAL SOURCE FILES / 官方下载文件
队列、训练集、验证集： 50000 35000 7500


## 亚组探索不等于外部验证
先报告各组人数和事件数，再看表现。年龄/IRSD亚组仍来自同一个模拟队列；两套数据资产也不是独立人群。稀疏亚组不强行输出完整指标。

In [2]:
from primecvd.survival import CoxModel
from primecvd.pipeline import subgroup_table
model = CoxModel(cfg["features"], alpha=0.0).fit(train)
display(subgroup_table(train, valid, model, cfg))

,subgroup,n,observed_events,events_by_horizon,early_censored,at_risk_after_horizon,horizon_years,mean_predicted_risk,KM_risk,KM_risk_ci_low,KM_risk_ci_high,Harrell_C,Uno_C_to_horizon,AUC_IPCW,Brier_IPCW,status
0,age_18_39,1707,10,10,988,709,5.0,0.005258,0.005916,0.003187,0.010968,0.777053,0.778649,0.765482,0.005832,exploratory; not external validation
1,age_40_59,4290,95,91,2439,1760,5.0,0.023990,0.021745,0.017733,0.026653,0.783861,0.775834,0.770248,0.020044,exploratory; not external validation
2,age_60_plus,1503,197,192,753,558,5.0,0.127895,0.132135,0.115598,0.150830,0.845544,0.842386,0.861544,0.072292,exploratory; not external validation
3,IRSD_1,1573,62,61,881,631,5.0,0.042849,0.040683,0.031703,0.052139,0.873807,0.878851,0.883052,0.027415,exploratory; not external validation
4,IRSD_2,1210,56,56,632,522,5.0,0.043775,0.046701,0.036128,0.060270,0.874914,0.876633,0.891221,0.029368,exploratory; not external validation
5,IRSD_3,1875,75,73,1053,749,5.0,0.044423,0.040255,0.032107,0.050417,0.857357,0.852266,0.855137,0.027013,exploratory; not external validation
6,IRSD_4,1269,54,50,730,489,5.0,0.036585,0.040340,0.030682,0.052956,0.905191,0.898157,0.893049,0.026577,exploratory; not external validation
7,IRSD_5,1573,55,53,884,636,5.0,0.034348,0.034998,0.026794,0.045655,0.870514,0.868794,0.867374,0.026422,exploratory; not external validation


## 受控的单位错误实验
固定验证人员及其所有其他Asset1特征，只替换HbA1c为“正确换算”和“故意遗漏换算”两个版本。它是在隔离数据质量影响，不是完全由EMR重建后的外部验证。原始文件保持不变。

In [3]:
from primecvd.data import load_emr
from primecvd.reconstruction import reconstruct
from primecvd.metrics import evaluate_survival
tables = load_emr(cfg)
correct, _, _ = reconstruct(tables, convert_units=True)
wrong, _, _ = reconstruct(tables, convert_units=False)
rows = []
for label, source in [("correct_conversion", correct), ("omitted_conversion", wrong)]:
    changed = valid.copy()
    changed["HbA1c"] = changed.Patient_ID.map(source.set_index("Patient_ID").HbA1c)
    result = evaluate_survival(train, changed, model.predict_risk(changed, cfg["horizon_years"]),
                               model.score(changed), cfg["horizon_years"], cfg["min_censor_survival"])
    result["scenario"] = label
    rows.append(result)
display(pd.DataFrame(rows))

,n,observed_events,events_by_horizon,early_censored,at_risk_after_horizon,horizon_years,mean_predicted_risk,KM_risk,KM_risk_ci_low,KM_risk_ci_high,Harrell_C,Uno_C_to_horizon,AUC_IPCW,Brier_IPCW,scenario
0,7500,302,293,4180,3027,5.0,0.040549,0.040297,0.035992,0.045106,0.875205,0.873691,0.876690,0.02728,correct_conversion
1,7500,302,293,4180,3027,5.0,0.082236,0.040297,0.035992,0.045106,0.842685,0.841378,0.844863,0.06424,omitted_conversion


## 独立练习
比较排序和校准相关指标是否以相同方向变化。不要把“误差变大”直接等同于“每个模型指标都必须变差”。说明本实验能回答什么、不能回答什么。

---
本周复盘：请用自己的话记录一个学会的概念、一个发现的错误和一个待解决问题。

特别提醒：不是寻找显著亚组；受控实验不会改动原始文件。